# 01 — Portfolio Overview

**Was Du hier siehst:** aktueller Stand des Portfolios in einer Ansicht — Positionen, Exposure pro Asset-Class und Currency, Marktwert-Trend der letzten 30 / 90 Tage, max-Drawdown.

**Datenquellen:**
- `pos_holdings` (letzter Import)
- `mkt_quotes_daily` (für Marktbewertung — `close` × `quantity`)
- `mkt_fx_daily` (Umrechnung in Portfolio-Currency, default EUR)
- `ref_instruments` (für `asset_type`)

**Annahmen:**
- Bewertung in EUR (anpassbar in Parameter-Zelle).
- Fehlende FX-Rate fällt zurück auf 1.0 mit Warning (kein Hard-Fail).
- Quote-Source-Priorität: `ib` vor `yfinance` (gleiche Logik wie monitor).

**Nicht enthalten:** Realisierte PnL, Cost-Basis-Korrekturen für Corporate Actions, Dividenden — das wäre Phase D.


In [ ]:
# Boilerplate — _db + _plot helpers laden, Style setzen.
import sys, pathlib
REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(REPO))

import pandas as pd
import numpy as np
from datetime import date, timedelta

from modules.analysis._db import session, df
from modules.analysis._plot import setup, COLORS, hline, plt
setup()

In [ ]:
# Parameter — anpassen falls nötig.
PORTFOLIO_CCY = 'EUR'    # alle Werte in dieser Währung
LOOKBACK_DAYS = 90       # MTM-Verlauf
QUOTE_SOURCE_PRIO = ['ib', 'yfinance']  # erstes Match gewinnt

## 1. Holdings + aktueller Spot

In [ ]:
with session() as con:
    holdings = df(con, '''
        SELECT h.holding_id, h.ref_instrument_id, h.quantity, h.cost_per_share,
               h.currency AS lot_currency, h.acquired_at, h.broker, h.account,
               i.symbol, i.name, i.asset_type, i.currency AS instr_currency,
               i.exchange
        FROM pos_holdings h
        LEFT JOIN ref_instruments i USING (ref_instrument_id)
        ORDER BY i.symbol
    ''')

    # Spot = letzter close pro instrument, mit Quote-Source-Prio.
    spots = df(con, '''
        WITH ranked AS (
            SELECT ref_instrument_id, ts, close, source,
                   ROW_NUMBER() OVER (
                       PARTITION BY ref_instrument_id
                       ORDER BY ts DESC,
                                CASE source WHEN 'ib' THEN 1 WHEN 'yfinance' THEN 2 ELSE 9 END
                   ) AS rk
            FROM mkt_quotes_daily
        )
        SELECT ref_instrument_id, ts AS spot_ts, close AS spot_close, source AS spot_source
        FROM ranked WHERE rk = 1
    ''')

holdings = holdings.merge(spots, on='ref_instrument_id', how='left')
print(f'Holdings: {len(holdings)}   Spots gefunden: {holdings.spot_close.notna().sum()}/{len(holdings)}')
holdings.head(20)

In [ ]:
# FX-Rates auf Portfolio-Currency (latest).
needed_currencies = set(holdings['instr_currency'].dropna().unique()) | {PORTFOLIO_CCY}
with session() as con:
    fx = df(con, '''
        WITH ranked AS (
            SELECT currency_from, currency_to, ts, rate,
                   ROW_NUMBER() OVER (PARTITION BY currency_from, currency_to ORDER BY ts DESC) AS rk
            FROM mkt_fx_daily
        )
        SELECT currency_from, currency_to, rate FROM ranked WHERE rk = 1
    ''')

def to_portfolio_ccy(value: float, from_ccy: str, fx_df: pd.DataFrame, target: str) -> float:
    if pd.isna(value) or value is None:
        return np.nan
    if from_ccy == target:
        return float(value)
    match = fx_df[(fx_df.currency_from == from_ccy) & (fx_df.currency_to == target)]
    if match.empty:
        return float(value)  # silent 1:1 fallback — wird unten geflaggt
    return float(value) * float(match.iloc[0]['rate'])

holdings['market_value_local'] = holdings['quantity'] * holdings['spot_close']
holdings['market_value_ptf']   = holdings.apply(
    lambda r: to_portfolio_ccy(r['market_value_local'], r['instr_currency'], fx, PORTFOLIO_CCY), axis=1)
holdings['cost_value_local']   = holdings['quantity'] * holdings['cost_per_share']
holdings['cost_value_ptf']     = holdings.apply(
    lambda r: to_portfolio_ccy(r['cost_value_local'], r['instr_currency'], fx, PORTFOLIO_CCY), axis=1)
holdings['pnl_ptf']            = holdings['market_value_ptf'] - holdings['cost_value_ptf']
holdings['pnl_pct']            = np.where(holdings['cost_value_ptf'] > 0,
                                           100.0 * holdings['pnl_ptf'] / holdings['cost_value_ptf'],
                                           np.nan)

# Audit: welche currencies haben keine FX-Rate (sind als 1:1 durchgereicht)?
missing_fx = sorted({c for c in holdings['instr_currency'].dropna().unique()
                     if c != PORTFOLIO_CCY and
                        fx[(fx.currency_from == c) & (fx.currency_to == PORTFOLIO_CCY)].empty})
if missing_fx:
    print(f'WARN: keine FX-Rate {missing_fx} -> {PORTFOLIO_CCY} — Werte als 1:1 behandelt.')

total_mv  = holdings['market_value_ptf'].sum(skipna=True)
total_cost= holdings['cost_value_ptf'].sum(skipna=True)
print(f'Total Market Value : {total_mv:>14,.2f} {PORTFOLIO_CCY}')
print(f'Total Cost Basis   : {total_cost:>14,.2f} {PORTFOLIO_CCY}')
print(f'Unrealized PnL     : {total_mv - total_cost:>14,.2f} {PORTFOLIO_CCY}'
      f'   ({100.0*(total_mv-total_cost)/total_cost:+.2f}%)' if total_cost > 0 else '')

## 2. Top-Positionen + PnL pro Position

In [ ]:
by_inst = (holdings
    .groupby(['ref_instrument_id', 'symbol', 'name', 'asset_type', 'instr_currency'], dropna=False)
    .agg(quantity=('quantity', 'sum'),
         market_value_ptf=('market_value_ptf', 'sum'),
         cost_value_ptf=('cost_value_ptf', 'sum'),
         pnl_ptf=('pnl_ptf', 'sum'))
    .assign(pnl_pct=lambda d: np.where(d.cost_value_ptf > 0,
                                        100.0 * d.pnl_ptf / d.cost_value_ptf, np.nan),
            weight_pct=lambda d: 100.0 * d.market_value_ptf / total_mv if total_mv > 0 else np.nan)
    .reset_index()
    .sort_values('market_value_ptf', ascending=False))

by_inst[['symbol', 'name', 'asset_type', 'instr_currency', 'quantity',
         'market_value_ptf', 'pnl_ptf', 'pnl_pct', 'weight_pct']].head(25)

In [ ]:
top = by_inst.head(15).copy()
if top.empty or top['market_value_ptf'].fillna(0).sum() == 0:
    print('Keine Top-Positionen — Bar-Plot übersprungen.')
else:
    fig, ax = plt.subplots(figsize=(10, 5.5))
    colors = [COLORS['positive'] if v >= 0 else COLORS['negative'] for v in top['pnl_ptf'].fillna(0)]
    ax.barh(top['symbol'][::-1], top['market_value_ptf'][::-1], color=colors[::-1])
    ax.set_title(f'Top 15 Positionen (Market Value in {PORTFOLIO_CCY}, Farbe = PnL-Sign)')
    ax.set_xlabel(f'Market Value ({PORTFOLIO_CCY})')
    plt.tight_layout(); plt.show()

## 3. Exposure: Asset-Class × Currency

In [ ]:
pivot = (holdings
    .assign(asset_type=holdings['asset_type'].fillna('unknown'),
            instr_currency=holdings['instr_currency'].fillna('???'))
    .pivot_table(index='asset_type', columns='instr_currency',
                 values='market_value_ptf', aggfunc='sum', fill_value=0.0))
pivot['_total'] = pivot.sum(axis=1)
pivot = pivot.sort_values('_total', ascending=False)
pivot_pct = 100.0 * pivot.drop(columns='_total') / total_mv if total_mv > 0 else pivot.drop(columns='_total')
print(f'Exposure in {PORTFOLIO_CCY} (Werte) und % vom Total:')
pivot

In [ ]:
plot_data = pivot.drop(columns='_total') if '_total' in pivot.columns else pivot
if plot_data.empty or plot_data.values.sum() == 0:
    print('Keine Positionen — Exposure-Plot übersprungen.')
else:
    fig, ax = plt.subplots(figsize=(10, 4.5))
    plot_data.plot(kind='bar', stacked=True, ax=ax, edgecolor='white', linewidth=0.5)
    ax.set_title(f'Exposure: Asset-Class × Currency ({PORTFOLIO_CCY})')
    ax.set_xlabel('Asset Type'); ax.set_ylabel(f'Market Value ({PORTFOLIO_CCY})')
    ax.legend(title='Currency', loc='upper right')
    plt.xticks(rotation=15); plt.tight_layout(); plt.show()

## 4. MTM-Trend (LOOKBACK_DAYS)

Tägliche Portfolio-Bewertung — `quantity` ist konstant (kein Trading), das Delta kommt rein aus `close` × FX. Fehlende Tage werden forward-filled (Wochenenden, US-Holidays etc.).

In [ ]:
since = date.today() - timedelta(days=LOOKBACK_DAYS)
with session() as con:
    q = df(con, '''
        WITH ranked AS (
            SELECT ref_instrument_id, ts, close, source,
                   ROW_NUMBER() OVER (
                       PARTITION BY ref_instrument_id, ts
                       ORDER BY CASE source WHEN 'ib' THEN 1 WHEN 'yfinance' THEN 2 ELSE 9 END
                   ) AS rk
            FROM mkt_quotes_daily
            WHERE ts >= ?
        )
        SELECT ref_instrument_id, ts, close FROM ranked WHERE rk = 1
    ''', [since])
    fx_hist = df(con, '''
        SELECT currency_from, currency_to, ts, rate FROM mkt_fx_daily
        WHERE ts >= ? AND currency_to = ?
    ''', [since, PORTFOLIO_CCY])

print(f'Quote-Punkte: {len(q):,}   FX-Punkte: {len(fx_hist):,}')

In [ ]:
# Long-form aufbauen: pro (instrument, tag) ein quote-Preis * quantity, in PORTFOLIO_CCY.
qty_lookup = holdings[['ref_instrument_id', 'quantity', 'instr_currency']].drop_duplicates(
    'ref_instrument_id', keep='last')
merged = q.merge(qty_lookup, on='ref_instrument_id', how='inner')
merged['mv_local'] = merged['quantity'] * merged['close']

fx_idx = fx_hist.rename(columns={'currency_from': 'instr_currency', 'rate': 'fx_rate'})
merged = merged.merge(fx_idx[['instr_currency', 'ts', 'fx_rate']], on=['instr_currency', 'ts'], how='left')
merged.loc[merged['instr_currency'] == PORTFOLIO_CCY, 'fx_rate'] = 1.0
merged['fx_rate'] = merged.groupby('instr_currency')['fx_rate'].ffill().bfill().fillna(1.0)
merged['mv_ptf'] = merged['mv_local'] * merged['fx_rate']

daily_total = (merged.groupby('ts')['mv_ptf'].sum()
                     .asfreq('D').ffill())  # Wochenenden
daily_total = daily_total.dropna()

if len(daily_total) < 2:
    print('Nicht genug Quote-History — MTM-Plot übersprungen.')
else:
    start = daily_total.iloc[0]
    end   = daily_total.iloc[-1]
    chg   = end - start
    chg_p = 100.0 * chg / start if start else 0
    print(f'{daily_total.index.min().date()} -> {daily_total.index.max().date()}'
          f'   {start:,.0f} -> {end:,.0f} {PORTFOLIO_CCY}   ({chg:+,.0f}, {chg_p:+.2f}%)')

    # Drawdown (laufendes Maximum).
    peak = daily_total.cummax()
    dd   = (daily_total - peak) / peak * 100.0
    max_dd = dd.min()
    max_dd_day = dd.idxmin()
    print(f'Max-Drawdown im Zeitraum: {max_dd:.2f}%  (am {max_dd_day.date()})')

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
    ax1.plot(daily_total.index, daily_total.values, color=COLORS['highlight'], linewidth=1.5)
    ax1.fill_between(daily_total.index, daily_total.values, alpha=0.08, color=COLORS['highlight'])
    hline(ax1, start, label='start')
    ax1.set_title(f'Portfolio MTM, letzte {LOOKBACK_DAYS} Tage  ({chg_p:+.2f}%)')
    ax1.set_ylabel(f'Market Value ({PORTFOLIO_CCY})')

    ax2.fill_between(dd.index, dd.values, 0, color=COLORS['negative'], alpha=0.4)
    ax2.plot(dd.index, dd.values, color=COLORS['negative'], linewidth=0.8)
    ax2.set_title(f'Drawdown vs. running peak (max {max_dd:.2f}%)')
    ax2.set_ylabel('Drawdown (%)'); ax2.set_xlabel('Datum')
    plt.tight_layout(); plt.show()

## 5. Stale-Quote-Check

Welche Holdings haben veraltete Quotes (= ingest-Lücke)? Schwelle: Spot-ts mehr als 5 Kalendertage in der Vergangenheit.

In [ ]:
if 'spot_ts' in holdings.columns:
    today = pd.Timestamp(date.today())
    h = holdings[['symbol', 'instr_currency', 'spot_ts', 'spot_source']].copy()
    h['spot_ts'] = pd.to_datetime(h['spot_ts'])
    h['days_stale'] = (today - h['spot_ts']).dt.days
    stale = h[h['days_stale'] > 5].sort_values('days_stale', ascending=False)
    if stale.empty:
        print('Alle Quotes sind frisch (≤ 5 Tage).')
    else:
        print(f'Stale-Quotes ({len(stale)}):')
    stale.head(20)

## 6. Per Portfolio-View

Aggregation pro Portfolio-Sicht (`list_portfolio_views`). Zeigt MV, Weight, PnL gruppiert nach den definierten Sichten. Wird nur sichtbar wenn das `portfolio_views`-Modul migriert + Sichten angelegt sind.

Konfigurations-Hinweis: pflege Sichten via `modules.db_edit` (CRUD auf `list_portfolio_views` + `list_portfolio_view_members`).

In [ ]:
# Per-View Aggregation
with session() as con:
    try:
        view_rows = df(con, '''
            SELECT v.view_id, v.name, v.description, v.color,
                   COUNT(DISTINCT m.lot_id) AS n_lots
            FROM list_portfolio_views v
            LEFT JOIN list_portfolio_view_members m USING (view_id)
            WHERE v.active = TRUE
            GROUP BY v.view_id, v.name, v.description, v.color
            ORDER BY v.view_id
        ''')
        # Member-Daten lot-level joinen
        view_members = df(con, '''
            SELECT m.view_id, h.ref_instrument_id, m.lot_id
            FROM list_portfolio_view_members m
            JOIN pos_holdings h USING (lot_id)
        ''')
    except Exception as e:
        print(f'Portfolio-Views nicht verfuegbar ({e.__class__.__name__}). Section uebersprungen.')
        view_rows = pd.DataFrame()
        view_members = pd.DataFrame()

if not view_rows.empty and not holdings.empty:
    # Aggregation pro view + ref_instrument_id (Multi-Lot rollt zusammen)
    if 'mv_ptf' not in pos.columns:
        pos['mv_ptf'] = pos['market_value_ptf']
    join = view_members.merge(
        pos[['ref_instrument_id', 'symbol', 'market_value_ptf', 'pnl_ptf']],
        on='ref_instrument_id', how='inner')
    agg = (join.groupby('view_id')
                .agg(n_lots=('lot_id', 'nunique'),
                     n_instruments=('ref_instrument_id', 'nunique'),
                     market_value_ptf=('market_value_ptf', 'sum'),
                     pnl_ptf=('pnl_ptf', 'sum'),
                     symbols=('symbol', lambda s: ', '.join(sorted(set(s)))))
                .reset_index())
    agg['weight_pct'] = 100.0 * agg['market_value_ptf'] / total_mv if total_mv > 0 else np.nan
    agg = agg.merge(view_rows[['view_id', 'name']], on='view_id', how='left')
    cols = ['view_id', 'name', 'n_lots', 'n_instruments', 'market_value_ptf', 'weight_pct', 'pnl_ptf', 'symbols']
    print(f'Portfolio-Views ({len(agg)}):')
    agg[cols]
elif not view_rows.empty:
    print(f'{len(view_rows)} Sichten definiert, aber keine Lots verlinkt.')
    view_rows
